In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
base_path = Path("data/concate_data")
green_path = base_path / "green_tripdata_2025_all.parquet"
yellow_path = base_path / "yellow_tripdata_2025_all.parquet"

green_df = pd.read_parquet(green_path)
yellow_df = pd.read_parquet(yellow_path)

print(f"Green shape: {green_df.shape}")
print(f"Yellow shape: {yellow_df.shape}")

Green shape: (543139, 21)
Yellow shape: (44417596, 20)


In [15]:
yellow_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [17]:
# Quick EDA: concise structure, quality, and key numeric checks
for name, df in [("green", green_df), ("yellow", yellow_df)]:
    print(f"\n=== {name.upper()} DATASET ===")
    print("Rows, Cols:", df.shape)
    
    dtype_counts = df.dtypes.value_counts()
    print("Dtype counts:")
    print(dtype_counts.to_string())
    
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    print("\nTop 5 columns by missing %:")
    print(missing_pct.head(5).round(2).to_string())
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    key_numeric = [c for c in [
        "trip_distance", "fare_amount", "tip_amount", "total_amount",
        "passenger_count", "tolls_amount"
    ] if c in numeric_cols]
    
    if key_numeric:
        print("\nKey numeric summary:")
        print(df[key_numeric].describe().round(2).to_string())
    
    print("\nSample rows:")
    print(df.head(2).to_string(index=False))


=== GREEN DATASET ===
Rows, Cols: (543139, 21)
Dtype counts:
float64           15
int32              3
datetime64[us]     2
object             1

Top 5 columns by missing %:
ehail_fee               100.00
trip_type                 8.12
congestion_surcharge      8.11
store_and_fwd_flag        8.11
RatecodeID                8.11

Key numeric summary:
       trip_distance  fare_amount  tip_amount  total_amount  passenger_count  tolls_amount
count      543139.00    543139.00   543139.00     543139.00        499102.00     543139.00
mean           18.43        18.18        2.67         25.23             1.29          0.27
std          1137.17        17.73        3.65         19.99             0.94          1.44
min             0.00      -470.60     -100.00       -473.10             0.00         -6.94
25%             1.21         9.30        0.00         14.64             1.00          0.00
50%             1.98        13.58        2.08         20.02             1.00          0.00
75%        

In [5]:
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_error

# Build yellow-only modeling frame and target duration in minutes
yellow_model_df = yellow_df.copy()
yellow_model_df["duration"] = (
    (yellow_model_df["tpep_dropoff_datetime"] - yellow_model_df["tpep_pickup_datetime"]).dt.total_seconds() / 60
).round(2)

print(yellow_model_df["duration"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(2))

# Keep only valid durations for training
yellow_model_df = yellow_model_df[(yellow_model_df["duration"] > 0) & (yellow_model_df["duration"] <= 70)].copy()

# Restrict model inputs to only these features
feature_cols = ["PULocationID", "DOLocationID", "trip_distance"]
model_cols = feature_cols + ["duration"]
yellow_model_df = yellow_model_df[model_cols].dropna().copy()

# Quick-run sample so AutoGluon trains in reasonable time
sample_n = min(200000, len(yellow_model_df))
yellow_model_df = yellow_model_df.sample(n=sample_n, random_state=42).reset_index(drop=True)

print("Prepared yellow_model_df shape:", yellow_model_df.shape)
print("Training columns:", yellow_model_df.columns.tolist())
print("duration preview:")
print(yellow_model_df["duration"].describe().round(2))

c:\Users\Martin\anaconda3\envs\autogluon_win\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


count    44417596.00
mean           17.20
std            28.44
min        -51472.32
10%             4.83
25%             8.08
50%            13.38
75%            21.28
90%            32.33
95%            42.93
99%            69.90
max         14880.77
Name: duration, dtype: float64
Prepared yellow_model_df shape: (200000, 4)
Training columns: ['PULocationID', 'DOLocationID', 'trip_distance', 'duration']
duration preview:
count    200000.00
mean         16.34
std          11.67
min           0.02
25%           8.22
50%          13.40
75%          21.08
max          70.00
Name: duration, dtype: float64


In [6]:
# Train/test split (model fit will use train only)
train_df, test_df = train_test_split(yellow_model_df, test_size=0.2, random_state=42)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (160000, 4)
Test shape: (40000, 4)


In [23]:
label = "duration"
predictor = TabularPredictor(
    label=label,
    problem_type="regression",
    eval_metric="mae",
    path="AutogluonModels/yellow_duration_quick"
 )

# Fit only on train_df (AutoGluon internally creates validation data from train_df)
predictor.fit(
    train_data=train_df,
    presets=["medium_quality_faster_train", "optimize_for_deployment"],
    # time_limit=300,
    auto_stack=False,
    num_bag_folds=0,
    num_stack_levels=0,
    excluded_model_types=["KNN", "RF"],
    verbosity=2
)

Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.20
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.11.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       22.06 GB / 31.75 GB (69.5%)
Disk Space Avail:   153.20 GB / 441.83 GB (34.7%)
Presets specified: ['medium_quality_faster_train', 'optimize_for_deployment']
Using hyperparameters preset: hyperparameters='default'
	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon will save models to "c:\Users\Martin\Desktop\Gatech\6242\Project\CSE6242_Team82\AutogluonModels\yellow_duration_quick"
T

[1000]	valid_set's l1: 4.48599
[2000]	valid_set's l1: 4.42014
[3000]	valid_set's l1: 4.38514
[4000]	valid_set's l1: 4.36282
[5000]	valid_set's l1: 4.34119
[6000]	valid_set's l1: 4.32248
[7000]	valid_set's l1: 4.31181
[8000]	valid_set's l1: 4.3007
[9000]	valid_set's l1: 4.2921
[10000]	valid_set's l1: 4.28293


	-4.2829	 = Validation score   (-mean_absolute_error)
	12.46s	 = Training   runtime
	0.23s	 = Validation runtime
Fitting model: LightGBM ...
	Fitting with cpus=10, gpus=0, mem=0.0/22.0 GB


[1000]	valid_set's l1: 4.22897
[2000]	valid_set's l1: 4.16383
[3000]	valid_set's l1: 4.14895
[4000]	valid_set's l1: 4.13387
[5000]	valid_set's l1: 4.13423


	-4.13	 = Validation score   (-mean_absolute_error)
	4.37s	 = Training   runtime
	0.06s	 = Validation runtime
Fitting model: CatBoost ...
	Fitting with cpus=10, gpus=0
	-4.089	 = Validation score   (-mean_absolute_error)
	124.7s	 = Training   runtime
	0.01s	 = Validation runtime
Fitting model: ExtraTreesMSE ...
	Fitting with cpus=16, gpus=0, mem=0.7/22.3 GB
	-4.2385	 = Validation score   (-mean_absolute_error)
	2.25s	 = Training   runtime
	0.08s	 = Validation runtime
Fitting model: NeuralNetFastAI ...
	Fitting with cpus=10, gpus=0, mem=0.0/21.7 GB
	-9.5468	 = Validation score   (-mean_absolute_error)
	94.36s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting model: XGBoost ...
	Fitting with cpus=10, gpus=0
	-4.143	 = Validation score   (-mean_absolute_error)
	3.85s	 = Training   runtime
	0.01s	 = Validation runtime
Fitting model: NeuralNetTorch ...
	Fitting with cpus=10, gpus=0, mem=0.0/22.2 GB
c:\Users\Martin\anaconda3\envs\autogluon_win\lib\site-packages\sklearn\compose\_colu

[1000]	valid_set's l1: 4.17058
[2000]	valid_set's l1: 4.123
[3000]	valid_set's l1: 4.12182


	-4.1184	 = Validation score   (-mean_absolute_error)
	5.41s	 = Training   runtime
	0.05s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ...
	Fitting 1 model on all data | Fitting with cpus=16, gpus=0, mem=0.0/22.2 GB
	Ensemble Weights: {'CatBoost': 0.542, 'LightGBMLarge': 0.417, 'XGBoost': 0.042}
	-4.0485	 = Validation score   (-mean_absolute_error)
	0.04s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 1669.77s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 37714.8 rows/s (2496 batch size)
Deleting model LightGBMXT. All files under c:\Users\Martin\Desktop\Gatech\6242\Project\CSE6242_Team82\AutogluonModels\yellow_duration_quick\models\LightGBMXT will be removed.
Deleting model LightGBM. All files under c:\Users\Martin\Desktop\Gatech\6242\Project\CSE6242_Team82\AutogluonModels\yellow_duration_quick\models\LightGBM will be removed.
Deleting model ExtraTreesMSE. All files under c:\Users\Martin\Desktop\Gatech\

In [28]:
from pathlib import Path

def folder_size_mb(path):
    return sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file()) / (1024 ** 2)

model_dir = Path(predictor.path)
size_before = folder_size_mb(model_dir)
print(f"Model dir before save_space: {size_before:.2f} MB")

# Safe deployment compression (keeps inference behavior)
predictor.save_space(remove_data=True, remove_fit_stack=True)

size_after = folder_size_mb(model_dir)
print(f"Model dir after save_space:  {size_after:.2f} MB")
print(f"Saved: {size_before - size_after:.2f} MB ({(size_before - size_after) / max(size_before, 1e-9) * 100:.1f}%)")

Model dir before save_space: 49.35 MB
Model dir after save_space:  49.35 MB
Saved: 0.00 MB (0.0%)


In [25]:
# Predict on holdout test split
test_pred = predictor.predict(test_df.drop(columns=[label]))
test_mae = mean_absolute_error(test_df[label], test_pred)
print(f"Holdout Test MAE (minutes): {test_mae:.3f}")

Holdout Test MAE (minutes): 4.063


In [26]:
predictor.evaluate(test_df)

{'mean_absolute_error': -4.062545914719827,
 'root_mean_squared_error': -6.143104715128562,
 'mean_squared_error': -37.737735541034766,
 'r2': 0.7252967032133186,
 'pearsonr': 0.853218149008339,
 'median_absolute_error': -2.6073394203186036}

In [27]:
predictor.leaderboard(test_df)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-4.062546,-4.048504,mean_absolute_error,0.898093,0.066181,133.993659,0.006065,0.000000,0.035149,2,True,4
1,CatBoost,-4.079820,-4.089010,mean_absolute_error,0.066754,0.006412,124.697406,0.066754,0.006412,124.697406,1,True,1
2,LightGBMLarge,-4.163257,-4.118393,mean_absolute_error,0.650585,0.046326,5.412651,0.650585,0.046326,5.412651,1,True,3
3,XGBoost,-4.169523,-4.142965,mean_absolute_error,0.174690,0.013443,3.848452,0.174690,0.013443,3.848452,1,True,2


In [ ]:
# Optional: train a compact web-deployment model (single CatBoost)
deploy_path = "AutogluonModels/yellow_duration_webui_cat"
deploy_predictor = TabularPredictor(
    label=label,
    problem_type="regression",
    eval_metric="mae",
    path=deploy_path,
).fit(
    train_data=train_df,
    presets=["optimize_for_deployment"],
    hyperparameters={"CAT": {}},
    fit_weighted_ensemble=False,
    auto_stack=False,
    num_bag_folds=0,
    num_stack_levels=0,
    time_limit=180,
    verbosity=1,
 )

deploy_pred = deploy_predictor.predict(test_df.drop(columns=[label]))
deploy_mae = mean_absolute_error(test_df[label], deploy_pred)
deploy_size_mb = folder_size_mb(deploy_predictor.path)

print(f"Web model MAE (minutes): {deploy_mae:.3f}")
print(f"Web model size: {deploy_size_mb:.2f} MB")
print(f"Web model path: {deploy_predictor.path}")